# RNNs with PyTorch — Character-Level Language Model

An RNN processes a sequence one step at a time, maintaining a **hidden state** that summarises past context: `h_t = f(x_t, h_{t-1})`.

**Corpus**: *Alice's Adventures in Wonderland* (Lewis Carroll, 1865) via NLTK Gutenberg.

We train a character-level model: given the last 100 characters, predict the next one. After training we can generate new text by feeding predictions back as input.

**(SOLUTION)**

## Step 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from nltk.corpus import gutenberg
import matplotlib.pyplot as plt

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## Step 2: Load Corpus

In [ ]:
text = gutenberg.raw('carroll-alice.txt')[:80000]

char2idx  = {ch: i for i, ch in enumerate(sorted(set(text)))}
idx2char  = {i: ch for ch, i in char2idx.items()}
vocab_size = len(char2idx)

print(f'Characters : {len(text):,}')
print(f'Vocab size : {vocab_size}')
print(f'Sample     : {text[:120]!r}')

## Step 3: Create Training Sequences

In [ ]:
SEQ_LEN = 100
STRIDE  = 5

data   = [char2idx[c] for c in text]
X_seqs = [data[i:i+SEQ_LEN]   for i in range(0, len(data)-SEQ_LEN-1, STRIDE)]
y_seqs = [data[i+1:i+SEQ_LEN+1] for i in range(0, len(data)-SEQ_LEN-1, STRIDE)]

X_tensor = torch.tensor(X_seqs, dtype=torch.long)
y_tensor = torch.tensor(y_seqs, dtype=torch.long)
print(f'Sequences: {len(X_seqs):,}  X:{X_tensor.shape}  y:{y_tensor.shape}')

## Step 4: DataLoader

In [ ]:
train_loader = DataLoader(
    TensorDataset(X_tensor, y_tensor),
    batch_size=128, shuffle=True
)
print(f'Batches per epoch: {len(train_loader)}')

## Step 5: Define the Model

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=256, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embed_dim)
        self.rnn         = nn.RNN(embed_dim, hidden_size, num_layers,
                                  batch_first=True, dropout=0.3)
        self.fc          = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x      = self.embedding(x)
        out, h = self.rnn(x, hidden)
        return self.fc(out), h

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

model = CharRNN(vocab_size).to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## Step 6: Training Loop

In [ ]:
criterion   = nn.CrossEntropyLoss()
optimizer   = optim.Adam(model.parameters(), lr=0.002)
epoch_losses = []

for epoch in range(30):
    model.train()
    total_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        logits, _ = model(Xb)
        loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(train_loader)
    epoch_losses.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | Loss: {avg:.4f}')

## Step 7: Generate Text

In [ ]:
def generate(model, seed_text, char2idx, idx2char, length=400,
             temperature=0.8, device='cpu'):
    model.eval()
    with torch.no_grad():
        hidden = model.init_hidden(1, device)
        for ch in seed_text[:-1]:
            x = torch.tensor([[char2idx.get(ch, 0)]], dtype=torch.long, device=device)
            _, hidden = model(x, hidden)
        result  = seed_text
        current = seed_text[-1]
        for _ in range(length):
            x = torch.tensor([[char2idx.get(current, 0)]], dtype=torch.long, device=device)
            logits, hidden = model(x, hidden)
            probs  = torch.softmax(logits.squeeze() / temperature, dim=0)
            next_i = torch.multinomial(probs, 1).item()
            current = idx2char[next_i]
            result += current
    return result

print(generate(model, 'Alice ', char2idx, idx2char, 400, 0.8, str(device)))

## Step 8: Experiment with Temperature

In [ ]:
print('=== temperature=0.5 (focused) ===')
print(generate(model, 'The ', char2idx, idx2char, 200, 0.5, str(device)))

print('\n=== temperature=1.2 (creative) ===')
print(generate(model, 'The ', char2idx, idx2char, 200, 1.2, str(device)))

## Step 9: Visualize the RNN Architecture

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

chars = ['A', 'l', 'i', 'c', 'e', '...']
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(-1.2, 13.5)
ax.set_ylim(-0.5, 5.5)
ax.axis('off')
ax.set_title('Unrolled RNN — Character-by-Character Processing', fontsize=13, fontweight='bold')

xs = [i * 2.2 + 0.8 for i in range(len(chars))]
for i, (x, ch) in enumerate(zip(xs, chars)):
    # Input box
    r1 = mpatches.FancyBboxPatch((x - 0.5, 0.2), 1.0, 0.9, boxstyle='round,pad=0.05',
                                  facecolor='#74b9ff', edgecolor='#2d3436', lw=1.5)
    ax.add_patch(r1)
    ax.text(x, 0.65, f"x_{i}\n'{ch}'", ha='center', va='center', fontsize=9, fontweight='bold')

    # RNN cell
    r2 = mpatches.FancyBboxPatch((x - 0.6, 1.9), 1.2, 1.2, boxstyle='round,pad=0.05',
                                  facecolor='#a29bfe', edgecolor='#2d3436', lw=1.5)
    ax.add_patch(r2)
    ax.text(x, 2.5, f'RNN\nh_{i}', ha='center', va='center', fontsize=9, fontweight='bold')

    # Output box
    r3 = mpatches.FancyBboxPatch((x - 0.5, 3.7), 1.0, 0.9, boxstyle='round,pad=0.05',
                                  facecolor='#fd79a8', edgecolor='#2d3436', lw=1.5)
    ax.add_patch(r3)
    ax.text(x, 4.15, f"y_{i}", ha='center', va='center', fontsize=10, fontweight='bold')

    # Vertical arrows
    ax.annotate('', xy=(x, 1.9), xytext=(x, 1.1),
                arrowprops=dict(arrowstyle='->', color='#2d3436', lw=1.5))
    ax.annotate('', xy=(x, 3.7), xytext=(x, 3.1),
                arrowprops=dict(arrowstyle='->', color='#2d3436', lw=1.5))

    # Hidden state arrow
    if i > 0:
        ax.annotate('', xy=(x - 0.6, 2.5), xytext=(xs[i-1] + 0.6, 2.5),
                    arrowprops=dict(arrowstyle='->', color='#e17055', lw=2.5))

ax.text(-0.8, 0.65, 'Input', ha='center', fontsize=9, color='#636e72', style='italic')
ax.text(-0.8, 2.5,  'RNN\ncell', ha='center', fontsize=9, color='#636e72', style='italic')
ax.text(-0.8, 4.15, 'Output', ha='center', fontsize=9, color='#636e72', style='italic')
ax.text(xs[2], -0.3, 'Hidden state flows forward → (orange arrows)', ha='center',
        fontsize=9, color='#e17055', style='italic')
plt.tight_layout()
plt.show()


## Step 10: Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(epoch_losses)+1), epoch_losses, 'o-', color='#55efc4', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('RNN — Training Loss (Alice in Wonderland)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()